In [5]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from PIL import Image
from torchvision import transforms, models, datasets
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, confusion_matrix, classification_report

In [6]:
BASE_DATASET = r"Dataset"
EXTRACTED_PATH = r"processed_frames"
CELEB_PATH = r"Dataset\celeb DF V2"

IMG_SIZE = 256
FRAME_SKIP = 10
MAX_FRAMES = 12
BATCH_SIZE = 48
EPOCHS = 10
PATIENCE = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [7]:
def extract_frames_from_folder(input_folder, output_folder, label):

    os.makedirs(output_folder, exist_ok=True)

    for root, _, files in os.walk(input_folder):
        for file in tqdm(files):
            if not file.lower().endswith((".mp4", ".avi", ".mov")):
                continue

            video_path = os.path.join(root, file)
            cap = cv2.VideoCapture(video_path)

            frame_count = 0
            saved = 0

            while cap.isOpened() and saved < MAX_FRAMES:
                ret, frame = cap.read()
                if not ret:
                    break

                if frame_count % FRAME_SKIP == 0:
                    frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
                    save_name = f"{file}_{saved}.jpg"
                    save_path = os.path.join(output_folder, save_name)
                    cv2.imwrite(save_path, frame)
                    saved += 1

                frame_count += 1

            cap.release()

In [4]:
# Create folders
os.makedirs(f"{EXTRACTED_PATH}/train/real", exist_ok=True)
os.makedirs(f"{EXTRACTED_PATH}/train/fake", exist_ok=True)

# Extract Real
extract_frames_from_folder(
    os.path.join(BASE_DATASET, "Real"),
    f"{EXTRACTED_PATH}/train/real",
    0
)

# Extract Fake
extract_frames_from_folder(
    os.path.join(BASE_DATASET, "Fake"),
    f"{EXTRACTED_PATH}/train/fake",
    1
)

print("Frame extraction complete")

0it [00:00, ?it/s]
100%|██████████| 300/300 [00:24<00:00, 12.47it/s]
0it [00:00, ?it/s]
100%|██████████| 1000/1000 [04:58<00:00,  3.35it/s]

Frame extraction complete


In [8]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3,0.3,0.3,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

dataset = datasets.ImageFolder(f"{EXTRACTED_PATH}/train", transform_train)

targets = dataset.targets
class_counts = np.bincount(targets)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[t] for t in targets]

sampler = WeightedRandomSampler(sample_weights,
                                num_samples=len(sample_weights),
                                replacement=True)

train_loader = DataLoader(dataset,
                          batch_size=BATCH_SIZE,
                          sampler=sampler,
                          num_workers=4,
                          pin_memory=True)

print("Class counts:", class_counts)

Class counts: [38166  9446]


In [9]:
model = models.efficientnet_b0(weights="IMAGENET1K_V1")

for param in list(model.parameters())[:int(len(list(model.parameters()))*0.4)]:
    param.requires_grad = False

model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
model = model.to(device)

pos_weight = torch.tensor([class_counts[0]/class_counts[1]]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
best_f1 = 0
best_loss = float('inf')

early_loss_counter = 0
PATIENCE_LOSS = 3

temporary_models = []

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.4).float()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    train_loss = total_loss / len(train_loader)
    train_f1 = f1_score(all_labels, all_preds)

    scheduler.step()

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train F1:   {train_f1:.4f}")
    print(f"LR:         {optimizer.param_groups[0]['lr']:.6f}")

    # ------------------------------------------------
    # LOSS-BASED EARLY STOPPING
    # ------------------------------------------------
    if train_loss > best_loss:
        early_loss_counter += 1
        print(f"⚠️ Loss increased! Counter = {early_loss_counter}/{PATIENCE_LOSS}")

        if early_loss_counter >= PATIENCE_LOSS:
            print("🔄 Reloading last best_temp_model.pth due to loss increase!")
            model.load_state_dict(torch.load("best_temp_model.pth"))
            early_loss_counter = 0

    else:
        best_loss = train_loss
        early_loss_counter = 0

    # ------------------------------------------------
    # SAVE MODEL WHEN F1 IMPROVES
    # ------------------------------------------------
    if train_f1 > best_f1:
        best_f1 = train_f1

        filename = f"temp_model_epoch_{epoch + 1}.pth"
        torch.save(model.state_dict(), filename)
        temporary_models.append(filename)

        # Always store a "best so far" for loss fallback
        torch.save(model.state_dict(), "best_temp_model.pth")

        print(f"🔥 F1 improved → Saved {filename}")


# ============================================================
# AFTER TRAINING — EVALUATE ALL TEMP MODELS & PICK THE BEST
# ============================================================

print("\n🔍 Evaluating all temporary saved models to choose the BEST model...")

final_best_f1 = 0
final_best_model = None

for model_file in temporary_models:

    state = torch.load(model_file)
    model.load_state_dict(state)
    model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.4).float()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds)
    print(f"{model_file} → F1 = {f1:.4f}")

    if f1 > final_best_f1:
        final_best_f1 = f1
        final_best_model = model_file

# ------------------------------------------------------------
# SAVE ABSOLUTE BEST MODEL
# ------------------------------------------------------------
if final_best_model is not None:
    torch.save(torch.load(final_best_model), "best_model.pth")
    print(f"\n🏆 BEST MODEL SELECTED: {final_best_model}")
    print("💾 Saved final model as: best_model.pth")
else:
    print("⚠️ No models were saved during training!")


Epoch 1/10


100%|██████████| 992/992 [02:32<00:00,  6.51it/s]


Train Loss: 0.7187
Train F1:   0.7913
LR:         0.000293
🔥 F1 improved → Saved temp_model_epoch_1.pth

Epoch 2/10


100%|██████████| 992/992 [02:32<00:00,  6.50it/s]


Train Loss: 0.5317
Train F1:   0.8474
LR:         0.000271
🔥 F1 improved → Saved temp_model_epoch_2.pth

Epoch 3/10


...

100%|██████████| 992/992 [02:33<00:00,  6.46it/s]


Train Loss: 0.2308
Train F1:   0.9321
LR:         0.000029
🔥 F1 improved → Saved temp_model_epoch_8.pth

Epoch 9/10


100%|██████████| 992/992 [02:31<00:00,  6.55it/s]


Train Loss: 0.2082
Train F1:   0.9382
LR:         0.000007
🔥 F1 improved → Saved temp_model_epoch_9.pth

Epoch 10/10


 69%|██████▉   | 684/992 [01:48<00:44,  6.88it/s]

In [ ]:
def predict_video_simple(video_path):

    cap = cv2.VideoCapture(video_path)
    probs = []
    count = 0

    while cap.isOpened() and len(probs) < 5:
        ret, frame = cap.read()
        if not ret:
            break

        if count % 20 == 0:
            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            tensor = transform_val(image).unsqueeze(0).to(device)

            with torch.no_grad():
                output = model(tensor)
                prob = torch.sigmoid(output).item()
                probs.append(prob)

        count += 1

    cap.release()

    if len(probs) == 0:
        return 0.0

    return np.mean(probs)

In [ ]:
def test_celebdf():

    model.load_state_dict(torch.load("best_model.pth"))
    model.eval()

    all_preds = []
    all_labels = []

    celeb_real_path = os.path.join(CELEB_PATH, "Celeb-real")
    youtube_real_path = os.path.join(CELEB_PATH, "YouTube-real")
    celeb_fake_path = os.path.join(CELEB_PATH, "Celeb-synthesis")

    # ---------- REAL FOLDERS ----------
    for folder in [celeb_real_path, youtube_real_path]:
        for file in os.listdir(folder):
            if not file.lower().endswith((".mp4", ".avi", ".mov")):
                continue

            video_path = os.path.join(folder, file)
            final_prob = predict_video_simple(video_path)

            pred = 1 if final_prob > 0.4 else 0
            all_preds.append(pred)
            all_labels.append(0)

    # ---------- FAKE FOLDER ----------
    for file in os.listdir(celeb_fake_path):
        if not file.lower().endswith((".mp4", ".avi", ".mov")):
            continue

        video_path = os.path.join(celeb_fake_path, file)
        final_prob = predict_video_simple(video_path)

        pred = 1 if final_prob > 0.4 else 0
        all_preds.append(pred)
        all_labels.append(1)

    print("\n===== CELEB-DF V2 RESULTS =====")
    print(confusion_matrix(all_labels, all_preds))
    print(classification_report(all_labels, all_preds))

In [ ]:
test_celebdf()

NameError: name 'model' is not defined

In [ ]:
import os

for file in os.listdir():
    if file.endswith(".pth") and file != "best_model.pth":
        os.remove(file)
        print(f"Deleted: {file}")

print("All temporary .pth files removed.")

Deleted: best_temp_model.pth
Deleted: temp_model_epoch_1.pth
Deleted: temp_model_epoch_2.pth
Deleted: temp_model_epoch_3.pth
All temporary .pth files removed.
